# Parkinson's Disease Voice Classification

End-to-end ML pipeline: data loading → cleaning → EDA → feature engineering → hyperparameter tuning → evaluation → artifact export.

**Dataset:** UCI Parkinson's Dataset ()  
**Target:**  (0 = Healthy, 1 = Parkinson's)  
**Models:** Random Forest · Logistic Regression

## 1. Imports & Configuration

In [ ]:
import os
import time
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import psutil

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report,
)
from sklearn.model_selection import (
    GridSearchCV, learning_curve, train_test_split,
)
from sklearn.preprocessing import StandardScaler

# ── Configuration ─────────────────────────────────────────────────────────
DATASET_PATH  = "parkinsons.data"
TARGET_COLUMN = "status"
OUTPUT_DIR    = "parkinsons_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Parkinson's Disease Classification Pipeline")
print(f"Output directory: {OUTPUT_DIR}")


## 2. Data Loading

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load the dataset and print a basic summary."""
    df = pd.read_csv(path)

    print(f"Shape : {df.shape}")
    print(f"
Data types:
{df.dtypes.value_counts()}")
    print(f"
First 5 rows:")
    display(df.head())

    return df


df = load_data(DATASET_PATH)


## 3. Feature Engineering

Domain-derived features built **before** cleaning so they are available  
for EDA and correlation-based selection downstream.

In [ ]:
def create_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add interaction and aggregate features derived from vocal measures."""
    df_new = df.copy()

    jitter_cols  = [c for c in df.columns if "Jitter"  in c or "jitter"  in c.lower()]
    shimmer_cols = [c for c in df.columns if "Shimmer" in c or "shimmer" in c.lower()]

    if jitter_cols and shimmer_cols:
        df_new["jitter_shimmer_product"]   = df[jitter_cols].mean(axis=1) * df[shimmer_cols].mean(axis=1)
        df_new["avg_jitter"]               = df[jitter_cols].mean(axis=1)
        df_new["avg_shimmer"]              = df[shimmer_cols].mean(axis=1)
        df_new["voice_perturbation_index"] = (df[jitter_cols].mean(axis=1) + df[shimmer_cols].mean(axis=1)) / 2

    if "HNR" in df.columns and "NHR" in df.columns:
        df_new["hnr_nhr_ratio"] = df["HNR"] / (df["NHR"] + 1e-10)

    if "MDVP:Fhi(Hz)" in df.columns and "MDVP:Flo(Hz)" in df.columns:
        df_new["frequency_range"] = df["MDVP:Fhi(Hz)"] - df["MDVP:Flo(Hz)"]

    if all(c in df.columns for c in ["MDVP:Fo(Hz)", "MDVP:Fhi(Hz)", "MDVP:Flo(Hz)"]):
        df_new["frequency_cv"] = (
            (df["MDVP:Fhi(Hz)"] - df["MDVP:Flo(Hz)"]) / (df["MDVP:Fo(Hz)"] + 1e-10)
        )

    if "spread1" in df.columns and "spread2" in df.columns:
        df_new["spread_interaction"] = df["spread1"] * df["spread2"]

    if "DFA" in df.columns and "PPE" in df.columns:
        df_new["dfa_ppe_product"] = df["DFA"] * df["PPE"]

    new_cols = [c for c in df_new.columns if c not in df.columns]
    print(f"Engineered {len(new_cols)} new features: {new_cols}")
    return df_new


df = create_engineered_features(df)
print(f"Shape after feature engineering: {df.shape}")


## 4. Exploratory Data Analysis

In [ ]:
def run_eda(df: pd.DataFrame, target_col: str = "status") -> None:
    """Generate and save key EDA visualisations."""
    eda_dir = f"{OUTPUT_DIR}/eda_plots"
    os.makedirs(eda_dir, exist_ok=True)

    # 1. Target distribution
    target_counts = df[target_col].value_counts().sort_index()
    plt.figure(figsize=(8, 5))
    bars = plt.bar(target_counts.index, target_counts.values,
                   color=["#3498db", "#e74c3c"], edgecolor="black", width=0.6)
    plt.title("Parkinson's Disease – Class Distribution", fontsize=14, fontweight="bold")
    plt.xlabel("Status (0 = Healthy, 1 = Parkinson's)")
    plt.ylabel("Count")
    plt.xticks([0, 1], ["Healthy", "Parkinson's"])
    for bar, (_, v) in zip(bars, target_counts.items()):
        plt.text(bar.get_x() + bar.get_width() / 2, v + len(df) * 0.01,
                 f"{v}\n({v / len(df) * 100:.1f}%)", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{eda_dir}/01_target_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 2. PPE scatterplot (most discriminative feature)
    if "PPE" in df.columns:
        healthy    = df[df[target_col] == 0]
        parkinsons = df[df[target_col] == 1]
        plt.figure(figsize=(12, 5))
        plt.scatter(healthy.index,    healthy["PPE"],    c="#3498db", label="Healthy",
                    alpha=0.6, s=50, edgecolors="black", linewidth=0.4)
        plt.scatter(parkinsons.index, parkinsons["PPE"], c="#e74c3c", label="Parkinson's",
                    alpha=0.6, s=50, edgecolors="black", linewidth=0.4)
        plt.title("PPE (Pitch Period Entropy) by Class", fontsize=14, fontweight="bold")
        plt.xlabel("Sample Index")
        plt.ylabel("PPE")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{eda_dir}/02_ppe_scatterplot.png", dpi=300, bbox_inches="tight")
        plt.show()
        print(f"PPE  |  Healthy mean: {healthy['PPE'].mean():.4f}  "
              f"Parkinson's mean: {parkinsons['PPE'].mean():.4f}")

    # 3. Key feature distributions
    key_features = ["MDVP:Fo(Hz)", "MDVP:Jitter(%)", "MDVP:Shimmer", "HNR", "RPDE", "DFA"]
    present      = [f for f in key_features if f in df.columns]
    fig, axes    = plt.subplots(2, 3, figsize=(15, 8))
    for ax, col in zip(axes.flatten(), present):
        ax.hist(df[col], bins=30, color="#3498db", alpha=0.7, edgecolor="black")
        ax.set_title(col, fontweight="bold")
        ax.set_xlabel("Value")
        ax.set_ylabel("Frequency")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{eda_dir}/03_feature_distributions.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 4. Box plots – features vs target
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, col in zip(axes.flatten(), present):
        data = [df[df[target_col] == 0][col].dropna(),
                df[df[target_col] == 1][col].dropna()]
        bp = ax.boxplot(data, labels=["Healthy", "Parkinson's"], patch_artist=True)
        for patch, color in zip(bp["boxes"], ["#3498db", "#e74c3c"]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title(col, fontweight="bold")
        ax.set_ylabel("Value")
        ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(f"{eda_dir}/04_boxplots_vs_target.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 5. Correlation heatmap (top features)
    numeric_df   = df.select_dtypes(include=[np.number])
    correlations = numeric_df.corr()[target_col].abs().sort_values(ascending=False)
    top_features = correlations.head(16).index.tolist()
    plt.figure(figsize=(12, 10))
    sns.heatmap(numeric_df[top_features].corr(), annot=True, fmt=".2f",
                cmap="RdYlBu_r", center=0, square=True, linewidths=1,
                cbar_kws={"shrink": 0.8})
    plt.title("Correlation Heatmap – Top 16 Features", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{eda_dir}/05_correlation_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"All EDA plots saved to: {eda_dir}/")


run_eda(df, TARGET_COLUMN)


## 5. Data Cleaning & Feature Selection

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Drop identifier column.
    2. Remove duplicates.
    3. Impute missing values with column median.
    4. Select the 15 features most correlated with the target.
    """
    print(f"Before cleaning: {df.shape}")

    # Drop non-feature identifier
    if "name" in df.columns:
        df = df.drop(columns=["name"])
        print("Dropped 'name' column")

    # Remove duplicates
    n_before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {n_before - len(df)} duplicate rows")

    # Impute missing values
    numeric_cols = [c for c in df.select_dtypes(include=["number"]).columns
                    if c != TARGET_COLUMN]
    missing = df[numeric_cols].isnull().sum()
    if missing.sum() > 0:
        print(f"Missing values found:
{missing[missing > 0]}")
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
        print("Imputed with column median")
    else:
        print("No missing values found")

    # Correlation-based feature selection (top 15)
    correlations = df.corr()[TARGET_COLUMN].abs().sort_values(ascending=False)
    correlations = correlations[correlations.index != TARGET_COLUMN]
    top_15 = correlations.head(15).index.tolist()

    print(f"
Top 15 features by |correlation| with '{TARGET_COLUMN}':")
    print("-" * 58)
    for i, feat in enumerate(top_15, 1):
        print(f"  {i:2d}. {feat:<35} {correlations[feat]:.4f}")

    n_dropped = len(correlations) - 15
    print(f"
Dropped {n_dropped} low-correlation features "
          f"({n_dropped / len(correlations) * 100:.1f}% reduction)")

    df_final = df[top_15 + [TARGET_COLUMN]]
    print(f"Final shape: {df_final.shape}")
    return df_final


df = clean_data(df)


## 6. Target Validation

In [ ]:
def validate_target(df: pd.DataFrame, target: str) -> None:
    """Sanity-check the target column before modelling."""
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found.")
    if df[target].nunique() < 2:
        raise ValueError("Target has fewer than 2 classes – classification impossible.")

    labels = {0: "Healthy", 1: "Parkinson's"}
    counts = df[target].value_counts().sort_index()
    print(f"Target: {target}  |  Classes: {df[target].nunique()}")
    print("
Class distribution:")
    for label, count in counts.items():
        print(f"  {labels.get(label, label):<15} {count:>4}  ({count / len(df) * 100:.1f}%)")


validate_target(df, TARGET_COLUMN)


## 7. Feature Scaling

In [ ]:
def encode_and_scale(df: pd.DataFrame, target: str):
    """Separate features / target and apply StandardScaler."""
    X = df.drop(columns=[target])
    y = df[target]

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print(f"Features  : {X.shape[1]}")
    print(f"Samples   : {X.shape[0]}")
    print("Scaling   : StandardScaler applied")
    return X, X_scaled, y, scaler


X, X_scaled, y, scaler = encode_and_scale(df, TARGET_COLUMN)


## 8. Train / Test Split

In [ ]:
def safe_train_test_split(X, y, test_size: float = 0.2):
    """Stratified train/test split."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    labels = {0: "Healthy", 1: "Parkinson's"}
    print(f"Train : {len(X_train)} samples  |  Test : {len(X_test)} samples")
    print(f"Split : {int((1 - test_size) * 100)}/{int(test_size * 100)}")
    print("
Training class distribution:")
    for label, count in y_train.value_counts().sort_index().items():
        print(f"  {labels.get(label, label):<15} {count:>4}  ({count / len(y_train) * 100:.1f}%)")

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = safe_train_test_split(X_scaled, y)


## 9. Hyperparameter Tuning

In [ ]:
def tune_hyperparameters(X_train, y_train) -> tuple:
    """Grid-search both models with regularisation focused on reducing overfitting."""
    tuned_models   = {}
    tuning_results = []

    # ── Random Forest ──────────────────────────────────────────────────────
    print("Tuning Random Forest...")
    rf_param_grid = {
        "n_estimators":        [50, 100],
        "max_depth":           [3, 5, 7],
        "min_samples_split":   [20, 30, 40],
        "min_samples_leaf":    [10, 15, 20],
        "max_features":        ["sqrt", "log2"],
        "min_impurity_decrease": [0.01, 0.02],
        "class_weight":        ["balanced"],
    }
    rf      = RandomForestClassifier(random_state=42, n_jobs=-1)
    rf_grid = GridSearchCV(rf, rf_param_grid, cv=5, scoring="f1", n_jobs=-1, verbose=0)

    t0 = time.time()
    rf_grid.fit(X_train, y_train)
    rf_time = time.time() - t0

    tuned_models["Random_Forest"] = rf_grid.best_estimator_
    print(f"  Best params : {rf_grid.best_params_}")
    print(f"  Best CV F1  : {rf_grid.best_score_:.4f}")
    print(f"  Time        : {rf_time:.1f}s")
    tuning_results.append({"Model": "Random_Forest", "Best_CV_Score": rf_grid.best_score_,
                            "Best_Params": rf_grid.best_params_, "Training_Time": rf_time})

    # ── Logistic Regression ────────────────────────────────────────────────
    print("
Tuning Logistic Regression...")
    lr_param_grid = {
        "C":           [0.001, 0.01, 0.1, 1.0],
        "penalty":     ["l1", "l2", "elasticnet"],
        "solver":      ["saga"],
        "l1_ratio":    [0.3, 0.5, 0.7],
        "class_weight": ["balanced"],
        "max_iter":    [2000],
    }
    lr      = LogisticRegression(random_state=42)
    lr_grid = GridSearchCV(lr, lr_param_grid, cv=5, scoring="f1", n_jobs=-1, verbose=0)

    t0 = time.time()
    lr_grid.fit(X_train, y_train)
    lr_time = time.time() - t0

    tuned_models["Logistic_Regression"] = lr_grid.best_estimator_
    print(f"  Best params : {lr_grid.best_params_}")
    print(f"  Best CV F1  : {lr_grid.best_score_:.4f}")
    print(f"  Time        : {lr_time:.1f}s")
    tuning_results.append({"Model": "Logistic_Regression", "Best_CV_Score": lr_grid.best_score_,
                            "Best_Params": lr_grid.best_params_, "Training_Time": lr_time})

    # Save summary
    pd.DataFrame(tuning_results).to_csv(f"{OUTPUT_DIR}/hyperparameter_tuning.csv", index=False)
    print(f"
Tuning results saved to {OUTPUT_DIR}/hyperparameter_tuning.csv")
    return tuned_models, tuning_results


models, tuning_results = tune_hyperparameters(X_train, y_train)


## 10. Model Evaluation

In [ ]:
def plot_learning_curve(model, X_train, y_train, model_name: str) -> None:
    """Plot and save a learning curve to diagnose bias/variance."""
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train,
        cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring="accuracy",
    )
    t_mean, t_std = train_scores.mean(1), train_scores.std(1)
    v_mean, v_std = val_scores.mean(1),   val_scores.std(1)

    plt.figure(figsize=(9, 5))
    plt.plot(train_sizes, t_mean, "o-", color="blue",  label="Training score")
    plt.fill_between(train_sizes, t_mean - t_std, t_mean + t_std, alpha=0.15, color="blue")
    plt.plot(train_sizes, v_mean, "s-", color="red",   label="CV score")
    plt.fill_between(train_sizes, v_mean - v_std, v_mean + v_std, alpha=0.15, color="red")
    plt.xlabel("Training set size")
    plt.ylabel("Accuracy")
    plt.title(f"Learning Curve – {model_name}", fontweight="bold")
    plt.legend(loc="best")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/learning_curve_{model_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


def evaluate_models(models: dict, X_train, X_test, y_train, y_test) -> tuple:
    """Evaluate each model, print metrics, plot learning curves, and return a summary."""
    results = []

    for name, model in models.items():
        print(f"
{" " + name + " ":=^60}")

        y_train_pred = model.predict(X_train)
        y_test_pred  = model.predict(X_test)
        train_acc    = accuracy_score(y_train, y_train_pred)
        test_acc     = accuracy_score(y_test,  y_test_pred)

        prec    = precision_score(y_test, y_test_pred, zero_division=0)
        rec     = recall_score(y_test,    y_test_pred, zero_division=0)
        f1      = f1_score(y_test,        y_test_pred, zero_division=0)
        roc_auc = (roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
                   if hasattr(model, "predict_proba") else None)

        cm = confusion_matrix(y_test, y_test_pred)

        print(f"  Train accuracy   : {train_acc:.4f}")
        print(f"  Test  accuracy   : {test_acc:.4f}")
        print(f"  Overfitting gap  : {train_acc - test_acc:.4f}")
        print(f"  Precision        : {prec:.4f}")
        print(f"  Recall           : {rec:.4f}")
        print(f"  F1 Score         : {f1:.4f}")
        if roc_auc:
            print(f"  ROC AUC          : {roc_auc:.4f}")
        print(f"
  Confusion Matrix (rows=Actual, cols=Predicted):")
        print(f"                Healthy  Parkinson's")
        print(f"  Healthy          {cm[0][0]:3d}        {cm[0][1]:3d}")
        print(f"  Parkinson's      {cm[1][0]:3d}        {cm[1][1]:3d}")

        plot_learning_curve(model, X_train, y_train, name)

        results.append({
            "Model":            name,
            "Train_Accuracy":   train_acc,
            "Test_Accuracy":    test_acc,
            "Overfitting_Gap":  train_acc - test_acc,
            "Precision":        prec,
            "Recall":           rec,
            "F1_Score":         f1,
            "ROC_AUC":          roc_auc or 0,
            "True_Negatives":   int(cm[0][0]),
            "False_Positives":  int(cm[0][1]),
            "False_Negatives":  int(cm[1][0]),
            "True_Positives":   int(cm[1][1]),
        })

    df_results = pd.DataFrame(results)
    df_results.to_csv(f"{OUTPUT_DIR}/model_metrics.csv", index=False)

    best_name = df_results.loc[df_results["Test_Accuracy"].idxmax(), "Model"]

    print("
" + "=" * 60)
    print("MODEL COMPARISON SUMMARY")
    print("=" * 60)
    print(f"  {"Model":<25} {"Train":<10} {"Test":<10} {"Gap":<10} {"F1":<10}")
    print("-" * 60)
    for _, row in df_results.iterrows():
        print(f"  {row['Model']:<25} {row['Train_Accuracy']:<10.4f} "
              f"{row['Test_Accuracy']:<10.4f} {row['Overfitting_Gap']:<10.4f} {row['F1_Score']:<10.4f}")
    print(f"
  Best model: {best_name}")

    return df_results, best_name


df_results, best_model_name = evaluate_models(
    models, X_train, X_test, y_train, y_test
)


## 11. Resource Usage

In [ ]:
def measure_resource_usage(models: dict, X_test) -> pd.DataFrame:
    """Measure per-model inference time and process memory footprint."""
    process  = psutil.Process()
    records  = []

    for name, model in models.items():
        t0             = time.time()
        model.predict(X_test)
        inference_time = time.time() - t0
        memory_mb      = process.memory_info().rss / (1024 ** 2)

        print(f"{name}")
        print(f"  Inference time : {inference_time:.4f}s")
        print(f"  Memory (RSS)   : {memory_mb:.1f} MB")
        records.append({"Model": name, "Inference_Time_s": inference_time,
                         "Memory_MB": memory_mb})

    df_resources = pd.DataFrame(records)
    df_resources.to_csv(f"{OUTPUT_DIR}/resource_usage.csv", index=False)
    print(f"
Saved: {OUTPUT_DIR}/resource_usage.csv")
    return df_resources


resource_df = measure_resource_usage(models, X_test)


## 12. Feature Importance

In [ ]:
def plot_feature_importance(model, feature_names: list, model_name: str) -> None:
    """Horizontal bar chart of top-15 feature importances (tree models only)."""
    if not hasattr(model, "feature_importances_"):
        print(f"{model_name} does not expose feature_importances_.")
        return

    importances = model.feature_importances_
    indices     = np.argsort(importances)[-15:]

    plt.figure(figsize=(10, 7))
    plt.barh(range(len(indices)), importances[indices], color="#3498db", alpha=0.8)
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel("Feature Importance")
    plt.title(f"Top 15 Feature Importances – {model_name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/feature_importance_{model_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


## 13. Save Artifacts

In [ ]:
def save_artifacts(models: dict, scaler, best_model_name: str,
                   feature_names: list) -> None:
    """Pickle all models, the scaler, the best model, and feature names."""
    for name, model in models.items():
        path = f"{OUTPUT_DIR}/{name}.pkl"
        with open(path, "wb") as f:
            pickle.dump(model, f)
        print(f"Saved: {path}")

        if name == "Random_Forest":
            plot_feature_importance(model, feature_names, name)

    with open(f"{OUTPUT_DIR}/scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)
    print(f"Saved: {OUTPUT_DIR}/scaler.pkl")

    with open(f"{OUTPUT_DIR}/best_model.pkl", "wb") as f:
        pickle.dump(models[best_model_name], f)
    print(f"Saved: {OUTPUT_DIR}/best_model.pkl  ({best_model_name})")

    with open(f"{OUTPUT_DIR}/feature_names.pkl", "wb") as f:
        pickle.dump(feature_names, f)
    print(f"Saved: {OUTPUT_DIR}/feature_names.pkl")


save_artifacts(models, scaler, best_model_name, X.columns.tolist())


## 14. Pipeline Summary

In [ ]:
print("Pipeline complete.")
print(f"
Outputs in: {OUTPUT_DIR}/")
print(f"  EDA plots        : {OUTPUT_DIR}/eda_plots/")
print(f"  Model metrics    : {OUTPUT_DIR}/model_metrics.csv")
print(f"  Tuning results   : {OUTPUT_DIR}/hyperparameter_tuning.csv")
print(f"  Resource usage   : {OUTPUT_DIR}/resource_usage.csv")
print(f"  Trained models   : {OUTPUT_DIR}/*.pkl")
print(f"
Best model: {best_model_name}")

display(df_results[[
    "Model", "Train_Accuracy", "Test_Accuracy",
    "Overfitting_Gap", "F1_Score", "ROC_AUC"
]].round(4))
